# 02. RAG w Google Colab: TF-IDF, embeddingi, ChromaDB i lokalny LLM

Ten notebook pokazuje pełną ścieżkę RAG:

1. dokumenty,
2. wyszukiwanie prostą metodą TF-IDF,
3. embeddingi semantyczne,
4. baza wektorowa ChromaDB,
5. odpowiedź lokalnego LLM-a na podstawie kontekstu.

**Zalecenie:** `Runtime → Change runtime type → T4 GPU`.

In [1]:
!pip -q install transformers accelerate sentencepiece sentence-transformers chromadb scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import shutil
import torch
import numpy as np

print("GPU dostępne:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU dostępne: True
GPU: NVIDIA GeForce RTX 5070


## 1. Dokument testowy: mini-regulamin kursu

In [3]:
regulamin = """
Regulamin zaliczenia kursu AI

1. Student może mieć maksymalnie dwie nieobecności.
2. Trzecia nieobecność wymaga wykonania zadania dodatkowego.
3. Projekt końcowy musi zostać oddany do 30 czerwca.
4. Projekt może być wykonany indywidualnie albo w parach.
5. Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
6. Plagiat powoduje brak zaliczenia.
7. Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
"""

with open("regulamin_mini.txt", "w", encoding="utf-8") as f:
    f.write(regulamin)

print("Utworzono plik regulamin_mini.txt")

Utworzono plik regulamin_mini.txt


In [7]:
with open("regulamin_mini.txt", "r", encoding="utf-8") as f:
    text = f.read()

chunks = [line.strip() for line in text.split(".") if line.strip()]

for i, chunk in enumerate(chunks):
    print(i, ":", chunk)

0 : Regulamin zaliczenia kursu AI

1
1 : Student może mieć maksymalnie dwie nieobecności
2 : 2
3 : Trzecia nieobecność wymaga wykonania zadania dodatkowego
4 : 3
5 : Projekt końcowy musi zostać oddany do 30 czerwca
6 : 4
7 : Projekt może być wykonany indywidualnie albo w parach
8 : 5
9 : Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod
10 : 6
11 : Plagiat powoduje brak zaliczenia
12 : 7
13 : Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia


## 2. Lista dokumentów dla RAG

In [8]:
documents = [
    "Student może mieć maksymalnie dwie nieobecności.",
    "Trzecia nieobecność wymaga wykonania zadania dodatkowego.",
    "Projekt końcowy musi zostać oddany do 30 czerwca.",
    "Projekt może być wykonany indywidualnie albo w parach.",
    "Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.",
    "Plagiat powoduje brak zaliczenia.",
    "Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia."
]

for i, d in enumerate(documents):
    print(i, d)

0 Student może mieć maksymalnie dwie nieobecności.
1 Trzecia nieobecność wymaga wykonania zadania dodatkowego.
2 Projekt końcowy musi zostać oddany do 30 czerwca.
3 Projekt może być wykonany indywidualnie albo w parach.
4 Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
5 Plagiat powoduje brak zaliczenia.
6 Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.


## 3. Mini RAG bez bazy wektorowej: TF-IDF

To jest uproszczony mechanizm: pytanie → podobieństwo słów → najlepszy fragment.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_tfidf(query, documents, n_results=2):
    vectorizer = TfidfVectorizer()
    doc_vectors = vectorizer.fit_transform(documents)
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, doc_vectors).flatten()
    best_indices = similarities.argsort()[::-1][:n_results]

    return [(documents[i], float(similarities[i])) for i in best_indices]

query = "Czy mogę użyć ChatGPT w projekcie?"
results = retrieve_tfidf(query, documents, n_results=3)

print("Pytanie:", query)
for doc, score in results:
    print(f"{score:.3f} | {doc}")

Pytanie: Czy mogę użyć ChatGPT w projekcie?
0.345 | Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
0.000 | Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
0.000 | Plagiat powoduje brak zaliczenia.


## 4. Ładowanie lokalnego LLM-a w Colabie

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)

# Uwaga: na GPU używamy float16, na CPU float32.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

print("Model załadowany:", MODEL_NAME)

/home/jakub/PycharmProjects/SWPS_2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Urządzenie: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 632.33it/s]


Model załadowany: Qwen/Qwen2.5-0.5B-Instruct


In [11]:
def ask_llm(prompt, system="Jesteś pomocnym asystentem dydaktycznym. Odpowiadasz po polsku, krótko i precyzyjnie.", max_new_tokens=250):
    """Prosta funkcja do rozmowy z lokalnym modelem uruchomionym w Colabie."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip()

## 5. Mini RAG TF-IDF + LLM

In [14]:
def rag_tfidf(query, documents, n_results=2):
    selected = retrieve_tfidf(query, documents, n_results=n_results)
    selected_docs = [doc for doc, score in selected]
    context = " ".join(selected_docs)

    prompt = f"""
Odpowiedz na pytanie wyłącznie na podstawie kontekstu.

KONTEKST:
{context}

PYTANIE:
{query}

Jeżeli w kontekście nie ma odpowiedzi, napisz:
"Nie wiem na podstawie dostarczonych dokumentów."
"""
    answer = ask_llm(prompt, max_new_tokens=180)
    return answer, selected

query = "Czy mogę oddać projekt po 30 czerwca?"
answer, selected = rag_tfidf(query, documents)

print("Znalezione fragmenty:")
for doc, score in selected:
    print(f"{score:.3f} | {doc}")

print("Odpowiedź:")
print(answer)

Znalezione fragmenty:
0.604 | Projekt końcowy musi zostać oddany do 30 czerwca.
0.169 | Projekt może być wykonany indywidualnie albo w parach.
Odpowiedź:
Nie wiem na podstawie dostarczonych dokumentów.


## 6. RAG z embeddingami semantycznymi i ChromaDB

In [15]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedding_model.encode(documents).tolist()

print("Liczba dokumentów:", len(documents))
print("Rozmiar embeddingu:", len(embeddings[0]))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4755.04it/s]


Liczba dokumentów: 7
Rozmiar embeddingu: 384


In [17]:
DB_PATH = "rag_db"

if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_or_create_collection("regulamin")

collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=embeddings
)

print("Dokumenty dodane do ChromaDB.")

Dokumenty dodane do ChromaDB.


In [18]:
def retrieve_chroma(query, collection, embedding_model, n_results=3):
    query_embedding = embedding_model.encode([query]).tolist()[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results["documents"][0]

query = "Czy wolno korzystać z ChatGPT podczas robienia projektu?"
retrieved = retrieve_chroma(query, collection, embedding_model)

print("Pytanie:", query)
print("Najbardziej pasujące fragmenty:")
for doc in retrieved:
    print("-", doc)

Pytanie: Czy wolno korzystać z ChatGPT podczas robienia projektu?
Najbardziej pasujące fragmenty:
- Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
- Projekt może być wykonany indywidualnie albo w parach.
- Plagiat powoduje brak zaliczenia.


## 7. Pełny RAG: ChromaDB + lokalny LLM

In [20]:
def rag_chroma(query, collection, embedding_model, n_results=3):
    retrieved_docs = retrieve_chroma(query, collection, embedding_model, n_results=n_results)
    context = " ".join(retrieved_docs)

    prompt = f"""
Odpowiedz na pytanie wyłącznie na podstawie kontekstu.

KONTEKST:
{context}

PYTANIE:
{query}

ZASADY:
- Nie dopowiadaj niczego spoza kontekstu.
- Jeżeli odpowiedzi nie ma w kontekście, napisz: "Nie wiem na podstawie dostarczonych dokumentów."
- Odpowiedz po polsku.
"""

    answer = ask_llm(prompt, max_new_tokens=200)
    return answer, retrieved_docs

queries = [
    "Czy student z trzema nieobecnościami może zaliczyć kurs?",
    "Czy projekt można robić w parach?",
    "Czy wolno używać ChatGPT?",
    "Czy za aktywność można dostać wyższą ocenę?",
    "Czy egzamin jest ustny?"
]

for q in queries:
    print("=" * 90)
    print("PYTANIE:", q)
    answer, docs = rag_chroma(q, collection, embedding_model)
    print("FRAGMENTY:")
    for d in docs:
        print("-", d)
    print("ODPOWIEDŹ:")
    print(answer)

PYTANIE: Czy student z trzema nieobecnościami może zaliczyć kurs?
FRAGMENTY:
- Student może mieć maksymalnie dwie nieobecności.
- Trzecia nieobecność wymaga wykonania zadania dodatkowego.
- Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
ODPOWIEDŹ:
Tak, student może zaliczyć kurs, jeśli jego ocena końcowa jest podzielona o pół stopnia.
PYTANIE: Czy projekt można robić w parach?
FRAGMENTY:
- Projekt może być wykonany indywidualnie albo w parach.
- Projekt końcowy musi zostać oddany do 30 czerwca.
- Plagiat powoduje brak zaliczenia.
ODPOWIEDŹ:
Tak, projekt może być wykonany w parach.
PYTANIE: Czy wolno używać ChatGPT?
FRAGMENTY:
- Użycie ChatGPT jest dozwolone, ale student musi rozumieć kod.
- Plagiat powoduje brak zaliczenia.
- Aktywność na zajęciach może podnieść ocenę końcową o pół stopnia.
ODPOWIEDŹ:
Nie wiem na podstawie dostarczonych dokumentów.
PYTANIE: Czy za aktywność można dostać wyższą ocenę?
FRAGMENTY:
- Aktywność na zajęciach może podnieść ocenę końcową o p

## Puenta dydaktyczna

RAG nie trenuje modelu. RAG dostarcza modelowi właściwy kontekst. To różnica między „model wie” a „model otrzymał źródło”.